# Phase 1: Industrial Time-Series Preprocessing (CMP 공정 센서 데이터 전처리)

이 노트북에서는 반도체 제조 공정 중 하나인 **CMP (Chemical Mechanical Planarization, 화학적 기계 평탄화)** 설비의 센서 시계열 데이터를 다룹니다. 물리적 연마 과정에서 필연적으로 발생하는 거친 센서 노이즈와 이상치, 그리고 네트워크 결실로 인한 결측치를 정교하게 처리하는 데이터 엔지니어링 기법을 학습합니다.

## 🎯 실습 목표
1. **반도체 CMP 설비 센서 시뮬레이터 구축**: 물리 센서 5종의 가상 시계열을 수식으로 구현하여 재현 가능한 데이터 확보
2. **이상치(Outlier) 및 결측치(Missing Value) 정제**: IQR 기법을 통한 이상치 탐지 및 선형 보간(Linear Interpolation) 처리
3. **노이즈 필터링**: 단순이동평균(SMA)과 1차원 **칼만 필터 (Kalman Filter)**를 직접 수학적으로 구현하여 노이즈 감쇄 비교
4. **시계열 Resampling**: 고주파(10Hz) 원본 데이터를 분석에 유용한 Rolling Window 다운샘플링 처리

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d

# 시각화 테마 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows 한글 폰트 대응
plt.rcParams['axes.unicode_minus'] = False

print("필수 라이브러리 로드 완료")

## 1. 반도체 CMP 설비 센서 시뮬레이터 구현
CMP 공정은 회전하는 플래튼(Platen)에 웨이퍼를 밀착시키고 화학 슬러리를 흘려 표면을 갈아내는 가혹한 공정입니다. 
설비 센서 5종(`압력`, `회전속도`, `슬러리 유량`, `진동`, `모터 전류`)을 정밀하게 모사하는 가상 데이터 제너레이터를 구현하여 분석을 시작하겠습니다.

In [ ]:
def generate_cmp_sensor_data(n_samples=5000, seed=42):
    np.random.seed(seed)
    time = np.linspace(0, 500, n_samples) # 500초 분량의 10Hz 데이터
    
    # 1. Spindle Speed (RPM) : 정상 거동 1500 RPM 내외, 마찰력에 따른 미세 변동
    spindle_speed = 1500 + 10 * np.sin(2 * np.pi * 0.01 * time) + np.random.normal(0, 5, n_samples)
    
    # 2. Chamber Pressure (Bar) : 3.5 Bar 부근 압력 제어
    pressure = 3.5 + 0.1 * np.cos(2 * np.pi * 0.005 * time) + np.random.normal(0, 0.05, n_samples)
    
    # 3. Slurry Flow Rate (L/min) : 0.2 L/min 정상 주입
    slurry_flow = 0.2 + 0.01 * np.sin(2 * np.pi * 0.02 * time) + np.random.normal(0, 0.003, n_samples)
    
    # 4. Spindle Motor Current (A) : 마찰 부하에 비례하는 전류값
    motor_current = 15.0 + 0.5 * pressure * (spindle_speed / 1500.0) + np.random.normal(0, 0.15, n_samples)
    
    # 5. Vibration (g) : 고주파 마찰 진동
    vibration = 0.5 + 0.05 * np.sin(2 * np.pi * 0.1 * time) + np.random.normal(0, 0.08, n_samples)
    
    df = pd.DataFrame({
        'timestamp': time,
        'spindle_speed': spindle_speed,
        'pressure': pressure,
        'slurry_flow': slurry_flow,
        'motor_current': motor_current,
        'vibration': vibration
    })
    
    # 인위적인 결측치(NaN) 주입 (통신 끊김 상황 모사)
    nan_indices = np.random.choice(n_samples, size=int(n_samples * 0.03), replace=False)
    df.loc[nan_indices, 'pressure'] = np.nan
    
    # 인위적인 이상치(Outlier) 주입 (센서 오작동 스파이크 모사)
    outlier_indices = np.random.choice(n_samples, size=int(n_samples * 0.005), replace=False)
    df.loc[outlier_indices, 'motor_current'] = df.loc[outlier_indices, 'motor_current'] * 2.5
    
    return df

df_raw = generate_cmp_sensor_data()
print(df_raw.head())
print("\n--- 결측치 수량 ---")
print(df_raw.isnull().sum())

## 2. 결측치(NaN) 및 이상치(Outlier) 정제
실제 가혹한 공정 현장의 센서는 튀는 값이나 끊기는 값이 매우 잦습니다. 이를 통계적으로 정제합니다.

In [ ]:
# 1. 결측치(NaN) 보간 처리 : 시계열적 흐름을 보존하기 위해 앞뒤 값을 이용한 Linear Interpolation 수행
df_clean = df_raw.copy()
df_clean['pressure'] = df_clean['pressure'].interpolate(method='linear')

# 2. 이상치(Outlier) 제거 : IQR(Interquartile Range) 기법 적용
Q1 = df_clean['motor_current'].quantile(0.25)
Q3 = df_clean['motor_current'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 이상치 판정된 행을 상/하한 임계값으로 Clipping(대체) 처리
df_clean['motor_current'] = np.clip(df_clean['motor_current'], lower_bound, upper_bound)

print(f"이상치 상한 기준: {upper_bound:.2f} A, 하한 기준: {lower_bound:.2f} A")
print("보간 및 결측치 처리 후 결측 상태:", df_clean.isnull().sum().sum())

## 3. 노이즈 필터링 (이동평균 & 1차원 칼만 필터)
센서에 낀 마찰 노이즈를 걷어내기 위해, 지연 현상이 있는 이동평균과 확률 모델 기반의 **칼만 필터**를 비교 구현합니다.

In [ ]:
# 1. 단순 이동 평균 (Simple Moving Average)
df_clean['vibration_sma_15'] = df_clean['vibration'].rolling(window=15, min_periods=1, center=True).mean()

# 2. 1차원 칼만 필터 직접 구현
def kalman_filter_1d(z, Q=1e-4, R=1e-2):
    """
    z: 센서 측정값 시계열 배열
    Q: 공정 오차 공분산 (Process Variance)
    R: 측정 오차 공분산 (Measurement Variance)
    """
    n = len(z)
    x_hat = np.zeros(n)      # 추정 상태값
    P = np.zeros(n)          # 오차 공분산
    x_hat_minus = np.zeros(n)
    P_minus = np.zeros(n)
    K = np.zeros(n)          # 칼만 이득 (Kalman Gain)
    
    # 초기 추정값 설정
    x_hat[0] = z[0]
    P[0] = 1.0
    
    for k in range(1, n):
        # 1. 시간 업데이트 (Prediction)
        x_hat_minus[k] = x_hat[k-1]
        P_minus[k] = P[k-1] + Q
        
        # 2. 측정 업데이트 (Correction)
        K[k] = P_minus[k] / (P_minus[k] + R)
        x_hat[k] = x_hat_minus[k] + K[k] * (z[k] - x_hat_minus[k])
        P[k] = (1 - K[k]) * P_minus[k]
        
    return x_hat

# vibration 센서에 칼만 필터 적용
df_clean['vibration_kalman'] = kalman_filter_1d(df_clean['vibration'].values, Q=1e-5, R=8e-3)

# 결과 시각화
plt.figure(figsize=(15, 6))
plt.plot(df_raw['timestamp'][:300], df_raw['vibration'][:300], label='Original (Noise)', alpha=0.4, color='gray')
plt.plot(df_clean['timestamp'][:300], df_clean['vibration_sma_15'][:300], label='Simple Moving Average (15)', color='orange', linewidth=2)
plt.plot(df_clean['timestamp'][:300], df_clean['vibration_kalman'][:300], label='Kalman Filter', color='teal', linewidth=2)
plt.title('CMP Vibration Sensor Noise Filtering Comparison', fontsize=14)
plt.xlabel('Timestamp (seconds)', fontsize=12)
plt.ylabel('Vibration (g)', fontsize=12)
plt.legend(fontsize=11)
plt.show()

## 4. 시계열 Rolling Window Resampling (다운샘플링)
실시간 10Hz 센서 데이터를 분석 서버에 1초 단위(1Hz)로 요약 보고하기 위한 다운샘플링 파이프라인을 구축합니다.

In [ ]:
# timestamp를 정수형 또는 시계열 시간 축으로 근사화하여 groupby를 통해 1초 단위 평균값 산출
df_clean['sec_bucket'] = np.floor(df_clean['timestamp'])

df_resampled = df_clean.groupby('sec_bucket').agg({
    'spindle_speed': 'mean',
    'pressure': 'mean',
    'slurry_flow': 'mean',
    'motor_current': 'mean',
    'vibration_kalman': 'mean'
}).reset_index()

print(f"--- 샘플 크기 축소 확인 ---")
print(f"원본 샘플 수: {len(df_clean)}행 ➔ 1Hz Resampled 샘플 수: {len(df_resampled)}행")
print(df_resampled.head())

# 전처리된 중간 데이터를 로컬 데이터 폴더에 저장 (다음 Phase 활용)
import os
os.makedirs('./data', exist_ok=True)
df_resampled.to_csv('./data/cmp_phase1_cleaned.csv', index=False)
print("\nPhase 1 전처리 데이터 ('./data/cmp_phase1_cleaned.csv') 저장 완료")